In [ ]:
!pip install transformers datasets torch huggingface_hub

In [ ]:
from huggingface_hub import login

login()  # This will prompt you for your Hugging Face credentials

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Load the Phi-2 model and tokenizer
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Ensure that the tokenizer has a padding token defined, if not already
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model configuration
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Set the pad_token_id in the configuration
# This is crucial because PhiConfig might not have it by default
if tokenizer.pad_token_id is not None:
    config.pad_token_id = tokenizer.pad_token_id
else:
    # Fallback if eos_token_id is also None, which is unlikely for most tokenizers
    # This part can be simplified if tokenizer.eos_token_id is guaranteed to exist after setting tokenizer.pad_token = tokenizer.eos_token
    config.pad_token_id = tokenizer.eos_token_id # Or tokenizer.unk_token_id if eos_token_id is also not suitable

# Load the model with the updated configuration
model = AutoModelForCausalLM.from_pretrained(model_name, config=config, torch_dtype=torch.bfloat16, trust_remote_code=True)

print("Model and tokenizer initialized successfully")

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and tokenizer initialized successfully


In [ ]:
from datasets import load_dataset

# Load the datasets
warmup_ds = load_dataset("Vinnnf/Hybrid-OpenThoughts2-1M-1.5B")
rl_ds = load_dataset("agentica-org/DeepScaleR-Preview-Dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/2280390 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/44 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

deepscaler.json:   0%|          | 0.00/21.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40315 [00:00<?, ? examples/s]

In [ ]:
from tqdm import tqdm
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Load the Phi-2 model and tokenizer
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Ensure that the tokenizer has a padding token defined, if not already
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model configuration
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Set the pad_token_id in the configuration
# This is crucial because PhiConfig might not have it by default
if tokenizer.pad_token_id is not None:
    config.pad_token_id = tokenizer.pad_token_id
else:
    # Fallback if eos_token_id is also None, which is unlikely for most tokenizers
    config.pad_token_id = tokenizer.eos_token_id

# Load the model with the updated configuration
model = AutoModelForCausalLM.from_pretrained(model_name, config=config, torch_dtype=torch.bfloat16, trust_remote_code=True)


# Load the datasets - This line is added to resolve the NameError for warmup_ds
# Ideally, the cell with ID gw4UHBrX4V7E should be executed before this cell.
warmup_ds = load_dataset("Vinnnf/Hybrid-OpenThoughts2-1M-1.5B")

def preprocess_data(dataset, batch_size=8):
    inputs = []
    labels = []

    # Batched tokenization: Iterate over the dataset in batches
    for i in tqdm(range(0, len(dataset), batch_size), desc="Tokenizing", position=0):
        batch = dataset[i:i + batch_size]

        # In HF Datasets, dataset[i:j] returns a dict of lists
        # We can access the columns directly from this dict
        prompts = batch["instruction"]
        responses = batch["output"]

        tokenized_prompts = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
        tokenized_responses = tokenizer(responses, return_tensors="pt", padding=True, truncation=True, max_length=512)

        # Append tokenized inputs and labels
        inputs.append(tokenized_prompts)
        labels.append(tokenized_responses['input_ids'])

    # Flatten lists of inputs and labels
    if inputs:
        inputs = {key: torch.cat([x[key] for x in inputs], dim=0) for key in inputs[0]}
    else:
        inputs = {}

    if labels:
        labels = torch.cat(labels, dim=0)
    else:
        labels = torch.empty(0)

    return inputs, labels

# Example usage:
train_inputs, train_labels = preprocess_data(warmup_ds['train'])

In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

# Create DataLoader for training data
train_data = list(zip(train_inputs, train_labels))
train_loader = DataLoader(train_data, batch_size=2, shuffle=True)

# Set up optimizer and loss function
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
model.train()
for epoch in range(6):  # Set number of epochs
    for batch in train_loader:
        inputs = {key: value.squeeze().to(model.device) for key, value in batch[0].items()}
        labels = batch[1].squeeze().to(model.device)

        # Forward pass
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

In [ ]:
instruction = "Please reason step by step, and put your final answer within \\boxed{}."
prompt = "The arithmetic mean of 7, 2, $x$ and 10 is 9. What is the value of $x$?"

messages = [
    {"role": "user", "content": f"{instruction}\n{prompt}"},
]

# Prepare the prompt text for generation
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Generate output from the trained model
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384,
    do_sample=True,
    temperature=0.6,
    top_p=0.95
)

# Extract and decode the generated output
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Print the generated response
print(text + response)

In [ ]:
model.save_pretrained("phi2_trained_model")
tokenizer.save_pretrained("phi2_trained_tokenizer")